# 01 — Tokenization from Scratch

**Companion to [Chapter 02](../02-tokenization.md).**

You will:
1. Implement Byte Pair Encoding (BPE) training from scratch
2. Encode and decode with your own tokenizer
3. Build the sliding-window dataloader that produces LLM training data

Only `torch` is required. `tiktoken` is optional (one cell uses it if present).

## 1. Why not just use words or characters?

Run this to see the trade-off that motivates subword tokenization.

In [1]:
text = "The tokenizer tokenizes tokens by tokenizing text into tokens."

word_vocab = sorted(set(text.lower().replace('.', '').split()))
char_vocab = sorted(set(text))

print(f"word-level : {len(word_vocab):3d} vocab entries, {len(text.split()):3d} tokens for this sentence")
print(f"char-level : {len(char_vocab):3d} vocab entries, {len(text):3d} tokens for this sentence")
print()
print("word vocab:", word_vocab)
print()
print("Notice: 'tokenizer', 'tokenizes', 'tokens', 'tokenizing' are 4 unrelated entries")
print("in the word vocabulary, even though they share the stem 'token'.")
print("A subword tokenizer would learn 'token' once and reuse it.")

word-level :   8 vocab entries,   9 tokens for this sentence
char-level :  17 vocab entries,  62 tokens for this sentence

word vocab: ['by', 'into', 'text', 'the', 'tokenizer', 'tokenizes', 'tokenizing', 'tokens']

Notice: 'tokenizer', 'tokenizes', 'tokens', 'tokenizing' are 4 unrelated entries
in the word vocabulary, even though they share the stem 'token'.
A subword tokenizer would learn 'token' once and reuse it.


## 2. BPE training: merge the most frequent pair, repeatedly

The algorithm in four lines:

1. Start with every character as its own token
2. Count all adjacent pairs
3. Merge the most frequent pair into one new token
4. Repeat

Watch `low` assemble itself out of `l`, `o`, `w`.

In [2]:
from collections import Counter


def get_pair_counts(tokens):
    return Counter(zip(tokens, tokens[1:]))


def merge_pair(tokens, pair, new_token):
    out, i = [], 0
    while i < len(tokens):
        if i < len(tokens) - 1 and (tokens[i], tokens[i + 1]) == pair:
            out.append(new_token)
            i += 2
        else:
            out.append(tokens[i])
            i += 1
    return out


corpus = "low low low lower lowest newer newer wider"
tokens = list(corpus.replace(" ", "_"))
merges = []

print("start:", tokens[:20], "...\n")

for step in range(8):
    counts = get_pair_counts(tokens)
    if not counts:
        break
    best, freq = counts.most_common(1)[0]
    if freq < 2:
        print(f"stopping: most frequent pair only appears {freq}x")
        break
    new_token = best[0] + best[1]
    tokens = merge_pair(tokens, best, new_token)
    merges.append((best, new_token))
    print(f"merge {step + 1}: {best[0]!r} + {best[1]!r} -> {new_token!r}   ({freq} occurrences)")

print("\nfinal token stream:")
print(tokens)

start: ['l', 'o', 'w', '_', 'l', 'o', 'w', '_', 'l', 'o', 'w', '_', 'l', 'o', 'w', 'e', 'r', '_', 'l', 'o'] ...

merge 1: 'l' + 'o' -> 'lo'   (5 occurrences)
merge 2: 'lo' + 'w' -> 'low'   (5 occurrences)
merge 3: '_' + 'low' -> '_low'   (4 occurrences)
merge 4: 'e' + 'r' -> 'er'   (4 occurrences)
merge 5: '_low' + '_low' -> '_low_low'   (2 occurrences)
merge 6: '_' + 'n' -> '_n'   (2 occurrences)
merge 7: '_n' + 'e' -> '_ne'   (2 occurrences)
merge 8: '_ne' + 'w' -> '_new'   (2 occurrences)

final token stream:
['low', '_low_low', '_low', 'er', '_low', 'e', 's', 't', '_new', 'er', '_new', 'er', '_', 'w', 'i', 'd', 'er']


## 3. A complete BPE tokenizer class

Now package it: train on a corpus, then encode and decode arbitrary text.

The key detail in `encode`: merges must be applied **in the order they were learned**.
That ordering is the tokenizer's entire learned state.

In [3]:
class SimpleBPE:
    def __init__(self):
        self.merges = []          # ordered list of (pair -> new token)
        self.vocab = {}           # token string -> id

    def train(self, text, num_merges=50):
        tokens = list(text)
        for _ in range(num_merges):
            counts = get_pair_counts(tokens)
            if not counts:
                break
            best, freq = counts.most_common(1)[0]
            if freq < 2:
                break
            new_token = best[0] + best[1]
            tokens = merge_pair(tokens, best, new_token)
            self.merges.append((best, new_token))

        symbols = sorted(set(list(text)) | {new for _, new in self.merges})
        self.vocab = {s: i for i, s in enumerate(symbols)}
        self.inv_vocab = {i: s for s, i in self.vocab.items()}
        return self

    def tokenize(self, text):
        tokens = list(text)
        for pair, new_token in self.merges:          # order matters
            tokens = merge_pair(tokens, pair, new_token)
        return tokens

    def encode(self, text):
        return [self.vocab[t] for t in self.tokenize(text) if t in self.vocab]

    def decode(self, ids):
        return "".join(self.inv_vocab[i] for i in ids)


sample = ("the cat sat on the mat. the cat ate the rat. "
          "a cat and a rat sat on a mat. the mat was flat. ") * 8

bpe = SimpleBPE().train(sample, num_merges=40)

print(f"vocabulary size: {len(bpe.vocab)}")
print(f"merges learned : {len(bpe.merges)}")
print()
print("first 12 merges:")
for (a, b), new in bpe.merges[:12]:
    print(f"   {a!r} + {b!r} -> {new!r}")

vocabulary size: 56
merges learned : 40

first 12 merges:
   'a' + 't' -> 'at'
   'at' + ' ' -> 'at '
   'e' + ' ' -> 'e '
   't' + 'h' -> 'th'
   'th' + 'e ' -> 'the '
   'at' + '.' -> 'at.'
   'at.' + ' ' -> 'at. '
   'c' + 'at ' -> 'cat '
   'a' + ' ' -> 'a '
   'at. ' + 'the ' -> 'at. the '
   's' + 'at ' -> 'sat '
   'sat ' + 'o' -> 'sat o'


In [4]:
test = "the cat sat on a flat mat"

tokens = bpe.tokenize(test)
ids = bpe.encode(test)

print("text     :", test)
print("tokens   :", tokens)
print("ids      :", ids)
print("decoded  :", bpe.decode(ids))
print()
print(f"{len(test)} characters -> {len(tokens)} tokens "
      f"(compression ratio {len(test) / len(tokens):.2f}x)")
assert bpe.decode(bpe.encode(test)) == test, "round-trip failed!"
print("\nround-trip: OK")

text     : the cat sat on a flat mat
tokens   : ['the ', 'cat sat on ', 'a ', 'f', 'l', 'at ', 'm', 'at']
ids      : [54, 11, 3, 39, 41, 5, 42, 4]
decoded  : the cat sat on a flat mat

25 characters -> 8 tokens (compression ratio 3.12x)

round-trip: OK


### Exercise

Change `num_merges` to 5, then to 200, and re-run the two cells above.

- Few merges: many short tokens, low compression, small vocabulary
- Many merges: whole words become single tokens, high compression, large vocabulary

That trade-off — vocabulary size against sequence length — is the entire tokenizer design decision.

## 4. Real tokenizers: the traps

If `tiktoken` is installed, this cell demonstrates the three behaviors that cause
real-world bugs. If not, it explains them anyway.

In [5]:
try:
    import tiktoken
    enc = tiktoken.get_encoding("gpt2")

    print("TRAP 1: leading whitespace changes the token\n")
    for s in ["hello", " hello", "Hello", " Hello"]:
        print(f"  {s!r:10} -> {enc.encode(s)}")

    print("\nTRAP 2: numbers split inconsistently\n")
    for n in ["1", "12", "123", "1234", "12345", "127", "128"]:
        print(f"  {n:>6} -> {enc.encode(n)}")

    print("\nTRAP 3: non-English text costs more\n")
    pairs = [("English", "The weather is nice today"),
             ("Emoji  ", "The weather is nice today 🌤️🌈"),
             ("Code   ", "def f(x):\n    return x**2")]
    for label, s in pairs:
        ids = enc.encode(s)
        print(f"  {label}: {len(s):3d} chars -> {len(ids):3d} tokens "
              f"({len(s)/len(ids):.1f} chars/token)")

except ImportError:
    print("tiktoken not installed - `pip install tiktoken` to run this cell.")
    print()
    print("The three traps it demonstrates:")
    print("  1. 'hello' and ' hello' are DIFFERENT tokens (never end a prompt with a space)")
    print("  2. numbers split unpredictably, which is why arithmetic is unreliable")
    print("  3. non-English text and emoji use far more tokens for the same meaning")

tiktoken not installed - `pip install tiktoken` to run this cell.

The three traps it demonstrates:
  1. 'hello' and ' hello' are DIFFERENT tokens (never end a prompt with a space)
  2. numbers split unpredictably, which is why arithmetic is unreliable
  3. non-English text and emoji use far more tokens for the same meaning


## 5. From tokens to training data

Training data for an LLM is just the token stream, offset by one position:

```
input:   [The] [cat] [sat] [on]
target:  [cat] [sat] [on]  [the]
```

That off-by-one shift is the entire supervision signal of LLM pretraining.
`max_length` sets the window; `stride` sets how far it advances each time.

In [6]:
import torch
from torch.utils.data import Dataset, DataLoader


class GPTDataset(Dataset):
    def __init__(self, token_ids, max_length, stride):
        self.inputs, self.targets = [], []
        for i in range(0, len(token_ids) - max_length, stride):
            self.inputs.append(torch.tensor(token_ids[i:i + max_length]))
            self.targets.append(torch.tensor(token_ids[i + 1:i + max_length + 1]))

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        return self.inputs[idx], self.targets[idx]


ids = bpe.encode(sample)
print(f"corpus is {len(ids)} tokens long\n")

for stride in (4, 2, 1):
    ds = GPTDataset(ids, max_length=4, stride=stride)
    print(f"max_length=4, stride={stride}: {len(ds)} training samples")

print("\nSmaller stride = more samples from the same text, but heavy repetition.")

corpus is 49 tokens long

max_length=4, stride=4: 12 training samples
max_length=4, stride=2: 23 training samples
max_length=4, stride=1: 45 training samples

Smaller stride = more samples from the same text, but heavy repetition.


In [7]:
dataset = GPTDataset(ids, max_length=6, stride=6)
loader = DataLoader(dataset, batch_size=2, shuffle=False, drop_last=True)

x, y = next(iter(loader))
print("inputs  shape:", x.shape)
print("targets shape:", y.shape)
print()
for row in range(x.shape[0]):
    print(f"sample {row}")
    print(f"  input  ids: {x[row].tolist()}")
    print(f"  target ids: {y[row].tolist()}")
    print(f"  input  txt: {bpe.decode(x[row].tolist())!r}")
    print(f"  target txt: {bpe.decode(y[row].tolist())!r}")
    print()
print("Confirm by eye: the target is the input shifted LEFT by exactly one token.")

inputs  shape: torch.Size([2, 6])
targets shape: torch.Size([2, 6])

sample 0
  input  ids: [54, 35, 47, 0, 39, 41]
  target ids: [35, 47, 0, 39, 41, 8]
  input  txt: 'the cat sat on the mat. the cat ate the rat. a cat and a rat sat on a mat. the mat was fl'
  target txt: 'cat sat on the mat. the cat ate the rat. a cat and a rat sat on a mat. the mat was flat. the '

sample 1
  input  ids: [8, 35, 47, 0, 39, 41]
  target ids: [35, 47, 0, 39, 41, 8]
  input  txt: 'at. the cat sat on the mat. the cat ate the rat. a cat and a rat sat on a mat. the mat was fl'
  target txt: 'cat sat on the mat. the cat ate the rat. a cat and a rat sat on a mat. the mat was flat. the '

Confirm by eye: the target is the input shifted LEFT by exactly one token.


## What you built

- BPE training: repeatedly merge the most frequent adjacent pair
- Encoding/decoding with a learned merge list
- The sliding-window dataloader that turns text into `(input, target)` pairs

**Next:** [02 — Attention from Scratch](02_attention_from_scratch.ipynb)